# VAID in five minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/solara-associates/vaid/blob/main/examples/quickstart/quickstart.ipynb)

A VAID is a portable identity bound to an action an autonomous agent takes.
By the end of this notebook you will have minted one, delegated a narrower one
from it, verified the whole chain the way a stranger would, and watched it refuse
three things it should refuse.

Nothing is written to disk and no network call is made after the install below.
The signing keys are generated in memory and discarded when the kernel stops.

In [ ]:
%pip install "vaid-mint>=0.9,<0.10"

## Step 0: stand up a mint

A mint holds one signing key, called the kernel key, and stamps its trust domain
into everything it issues. The one below is ephemeral: the key lives only as long
as this kernel. Its **public** half is the only thing a third party needs to check
anything that follows, and that is what makes VAID offline. There is no callback
to the issuer, ever.

Read the `.assuming_nothing_revoked()` before you copy that line anywhere. A fresh
issuer has no revocation store it can vouch for, so out of the box it answers
`unavailable` and `verify_vaid()` denies everything, including documents it just
minted. That is deliberate: a revocation answer you could not obtain must not read
as *nothing is revoked*. The call opts into the fail-open posture by name, which is
right for a quickstart on a laptop and wrong for anything that must survive a
restart.

In [ ]:
from datetime import datetime, timezone

from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PrivateKey
from vaid_pop import canonical_request_signing_bytes

from vaid_mint import (
    ChainVerification,
    InMemoryAudit,
    MintPop,
    MintService,
    PresentedBundle,
    ReferenceIssuer,
    RevocationStatus,
    VaidSeed,
    VaidVerdict,
    build_mint_pop_payload,
    has_capability,
    is_in_scope,
    verify_chain,
    verify_vaid_authenticity,
    verify_vaid_standing,
)
from vaid_mint.error import UnauthorizedError

issuer = ReferenceIssuer.ephemeral(24, "vaid.example").assuming_nothing_revoked()
mint = MintService(issuer, InMemoryAudit())
kernel_key = issuer.kernel_public_key()

print("trust domain      :", "vaid.example")
print("kernel public key :", kernel_key.hex()[:32] + f"... ({len(kernel_key)} bytes)")

### Two helpers

Every agent holds its own key and the mint never sees the private half. A delegated
child always registers a key it holds, and the mint will not take the registration
on trust: the holder signs a canonical payload naming the key and the exact
attributes being requested, and the mint rebuilds the same bytes and checks the
signature. Both sides go through `build_mint_pop_payload`, so the bytes cannot
drift apart.

In [ ]:
def holder_keypair():
    """A key the agent itself holds. The mint only ever sees the public half."""
    private = Ed25519PrivateKey.generate()
    return private, private.public_key().public_bytes_raw()


def prove_possession(seed, private, public_raw):
    """Sign the mint's proof-of-possession challenge with the holder's own key."""
    nonce = f"quickstart-{id(private):x}"
    issued_at = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
    payload = build_mint_pop_payload(
        seed, public_key_der=public_raw, nonce=nonce, issued_at=issued_at
    )
    signature = private.sign(canonical_request_signing_bytes(payload))
    return MintPop(nonce=nonce, issued_at=issued_at, signature=signature)

## Step 1: mint a root identity

A VAID is a signed statement about one agent: who it is, which tenant it acts for,
which resources it may touch (its scope boundary), what it may do to them (its
capability set), and when that authority runs out. This root is the widest authority
in the notebook. Everything after it can only be narrower.

In [ ]:
orchestrator_key, orchestrator_pub = holder_keypair()

root_seed = VaidSeed(
    agent_class="orchestrator",
    version="1.0.0",
    tenant_id="acme",
    scope_boundary=["data.acme"],
    capability_set=["read", "write"],
    public_key_der=orchestrator_pub,
)
root = mint.mint_root(
    root_seed, prove_possession(root_seed, orchestrator_key, orchestrator_pub)
)

for field in ("vaid_id", "agent_class", "tenant_id", "scope_boundary",
              "capability_set", "expires_at"):
    print(f"{field:15}: {root[field]}")

assert issuer.verify_vaid(root)
assert verify_vaid_authenticity(kernel_key, root)

## Step 2: delegate a narrower child

The orchestrator hands part of its authority to a worker. The mint checks that the
request is a subset of what the parent holds on every axis at once: same tenant,
scope inside the parent's scope, capabilities the parent actually has, and a
lifetime that cannot outlast the parent's. The child below can read one branch of
the data its parent could read and write to all of.

Note that `mint_child` returns a response object and not the document. Read
`.vaid` for the document. The two extra fields say whether the child's life was cut
short by its parent's rather than by the issuer's own policy, which is otherwise
invisible: an `expires_at` on its own looks like an ordinary expiry.

In [ ]:
worker_key, worker_pub = holder_keypair()

child_seed = VaidSeed(
    agent_class="report-worker",
    version="1.0.0",
    tenant_id="acme",
    parent_vaid=root["vaid_id"],
    scope_boundary=["data.acme.reports"],
    capability_set=["read"],
    public_key_der=worker_pub,
)
response = mint.mint_child(
    child_seed, root, prove_possession(child_seed, worker_key, worker_pub)
)
child = response.vaid

print("scope_boundary          :", child["scope_boundary"], " was", root["scope_boundary"])
print("capability_set          :", child["capability_set"], " was", root["capability_set"])
print("expires_at              :", child["expires_at"])
print("expiry_bounded_by_parent:", response.expiry_bounded_by_parent)
print("parent_expires_at       :", response.parent_expires_at)

assert issuer.verify_vaid(child)
assert child["expires_at"] <= root["expires_at"]

## Step 3: verify the chain as a stranger would

A child carries its own scope and capabilities, not its ancestors'. So checking the
signature alone cannot answer whether the authority was legitimately derived. For
that, the holder presents the ancestor documents alongside the leaf and the verifier
walks the chain, rechecking tenant, scope, capabilities and expiry at every hop.

The walk below uses the public key and the two documents and nothing else. No
network, and no cooperation from the mint. The verdict is an enum rather than a
boolean so that *attenuation satisfied* can never be confused with *attenuation
unverifiable*.

In [ ]:
bundle = PresentedBundle([root, child])
verdict = verify_chain(kernel_key, child, bundle)

print("documents presented:", len(bundle))
print("verdict            :", verdict.name, f"(code {verdict.value!r})")
print("attenuation proved :", verdict.is_attenuated())
print()
print("other verdicts it can return:")
for v in sorted(ChainVerification, key=lambda v: v.value):
    if v is not ChainVerification.ATTENUATED:
        print("  ", v.value)

assert verdict is ChainVerification.ATTENUATED

## Step 4: three denials, with the reason for each

Minting is the easy half. What makes an identity useful is what it refuses, so here
are the three refusals you will meet first. The first two are decisions your own
code makes at the moment it is about to act, which is why they come back as plain
booleans and the reason code is yours to name. The third is the mint refusing to
issue at all, and it raises.

### 4a. A resource outside the scope boundary

Containment is segment bounded and not a text prefix, so a sibling branch is never
inside its neighbour. Note that the parent could still reach this resource. The
child gave that up when it was delegated.

In [ ]:
resource = "data.acme.payroll"
allowed = is_in_scope(child, resource)

print("resource  :", resource)
print("held scope:", child["scope_boundary"])
print("decision  :", "ALLOWED" if allowed else "DENIED")
print("reason    : out_of_scope")
print("parent could reach it:", is_in_scope(root, resource))

assert not allowed
assert is_in_scope(root, resource)

### 4b. An action it holds no capability for

Scope answers *which resource*, capabilities answer *what may be done to it*. Both
have to say yes. They are checked separately because they fail for different reasons
and need different fixes.

In [ ]:
capability = "write"
held = has_capability(child, capability)

print("capability:", capability)
print("held caps :", child["capability_set"])
print("decision  :", "ALLOWED" if held else "DENIED")
print("reason    : missing_capability")
print("parent holds it:", has_capability(root, capability))

assert not held
assert has_capability(root, capability)

### 4c. A child trying to widen its parent's scope

The worker tries to delegate more than it holds. This is refused before any key work
happens, so a rejected delegation leaves no trace and burns nothing.

The mint refusing is the convenient half. The part that matters is that the same
containment rule is applied again by the chain verifier in step 3: a widened
document minted by some other, laxer mint still fails verification at the hop that
widened it, as `not_attenuated`.

In [ ]:
grandchild_key, grandchild_pub = holder_keypair()

widening_seed = VaidSeed(
    agent_class="over-reaching-worker",
    version="1.0.0",
    tenant_id="acme",
    parent_vaid=child["vaid_id"],
    scope_boundary=["data.acme"],
    capability_set=["read"],
    public_key_der=grandchild_pub,
)

print("requested scope:", widening_seed.scope_boundary)
print("worker's scope :", child["scope_boundary"])

denied = None
try:
    mint.mint_child(
        widening_seed,
        child,
        prove_possession(widening_seed, grandchild_key, grandchild_pub),
    )
    print("decision       : ALLOWED")
except UnauthorizedError as exc:
    denied = str(exc)
    print("decision       : DENIED (UnauthorizedError)")
    print("reason         : scope_exceeds_parent")
    print("message        :", denied)

assert denied is not None

## Step 5: a revoked identity, through the RevocationCheck seam

Revocation is deliberately not part of the offline check, because an offline verifier
has nothing to consult. It is a seam: you plug in something that answers revoked, not
revoked, or unavailable, and verification uses that answer. Three states and not two,
because *I could not reach the list* must not read as *nothing is revoked*, and an
unavailable answer fails closed.

Watch the last two lines of the output together. The document is still authentic: it
was genuinely issued and its signature is intact, and revoking it cannot change that.
Authenticity and standing are different questions. Read a signature check as
*genuinely issued*, never as *currently authorised*.

In [ ]:
issuer.revoke(child["vaid_id"])
status = issuer.revocation_status(child)
standing = verify_vaid_standing(kernel_key, child, status)

print("revoked          :", child["vaid_id"])
print("revocation status:", status.value)
print("standing verdict :", standing.name, f"(code {standing.value!r})")
print("still authentic? :", verify_vaid_authenticity(kernel_key, child))

assert status is RevocationStatus.REVOKED
assert standing is VaidVerdict.REVOKED
assert verify_vaid_authenticity(kernel_key, child)

## Done

Revoking a parent revokes everything attenuated from it, so you do not have to
enumerate a delegation tree to shut it down. Try it: revoke `root` instead and watch
the child go with it.

Where to go next:

* [`docs/spec/encoding.md`](https://github.com/solara-associates/vaid/blob/main/docs/spec/encoding.md) for the rules that decide the signed bytes
* [`docs/spec/scope.md`](https://github.com/solara-associates/vaid/blob/main/docs/spec/scope.md) for how scope containment is decided
* [`docs/spec/revocation.md`](https://github.com/solara-associates/vaid/blob/main/docs/spec/revocation.md) for the `RevocationCheck` seam, and why it sits outside the conformance surface
* `npx vaid-skill` to get the same four verbs inside a coding agent